In [18]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.utils import check_X_y, _safe_indexing
from sklearn.preprocessing import LabelEncoder
import igraph as ig
import leidenalg as la
import numpy as np
from sklearn.neighbors import kneighbors_graph

In [19]:
def davies_bouldin_score_custom(X, labels, metric='euclidean', **kwds):
    # standard boilerplate for validation
    X, labels = check_X_y(X, labels)
    le = LabelEncoder()
    labels = le.fit_transform(labels)
    n_samples, n_features = X.shape
    n_labels = len(le.classes_)
    
    # ... (insert check_number_of_labels here)

    intra_dists = np.zeros(n_labels)
    centroids = np.zeros((n_labels, n_features))
    
    for k in range(n_labels):
        cluster_k = _safe_indexing(X, np.where(labels == k)[0])
        centroid = np.mean(cluster_k, axis=0)
        centroids[k] = centroid
        
        # Pass the metric to intra-cluster distance calculation
        # Note: we compare cluster points to the single centroid
        intra_dists[k] = np.mean(
            pairwise_distances(cluster_k, [centroid], metric=metric, **kwds)
        )

    # Pass the metric to inter-centroid distance calculation
    centroid_distances = pairwise_distances(centroids, metric=metric, **kwds)

    # Handle division by zero/edge cases
    if np.allclose(intra_dists, 0) or np.allclose(centroid_distances, 0):
        return 0.0

    # Fill diagonal with inf to ignore self-comparison in the max() step
    np.fill_diagonal(centroid_distances, np.inf)
    
    combined_intra_dists = intra_dists[:, None] + intra_dists
    scores = np.max(combined_intra_dists / centroid_distances, axis=1)
    
    return np.mean(scores)

In [20]:
base_path = "/home/wollerf/Projects/graph_based_embeddings.git/data/preprocessed/graph_based"
methods = ["UMAP", "PCA", "tSNE"]
datasets = ["HANCOCK", "MIMIC", "TCGA_LUAD"]
dims = [16, 32, 64]
NUM_RUNS = 10
result_dict = {"run": [], "dim": [], "method": [], "silhouette": [], "davies_bouldin": [], "dataset": []}

In [21]:
for data in datasets:
    print("Processing dataset = ", data)
    for dim in dims:
        print("Processing dimension = ", dim)
        for run in range(NUM_RUNS):
            vis_path = os.path.join(base_path, data, "visualization", f"{data}_samples_{dim}_{run}.tsv")
            vis_df = pd.read_csv(vis_path, sep='\t', index_col=0)
            vis_df.index = vis_df.index.astype("str")
            embedding_path = os.path.join(base_path, data, "embeddings", f"{data}_samples_{dim}_{run}.tsv")
            embedding_df = pd.read_csv(embedding_path, sep='\t', index_col=0)
            embedding_df.index = embedding_df.index.astype("str")
            
            # Cluster based on high-dimensional data.
            embedding_mat = embedding_df.to_numpy()
            # --- Parameters for KNN ---
            n_neighbors = int(0.05 * len(vis_df))

            pome_adj = kneighbors_graph(
                embedding_mat,
                n_neighbors=n_neighbors,
                mode='connectivity',
                include_self=False,
                metric='sqeuclidean'
            )

            # Convert sparse matrix to igraph
            sources, targets = pome_adj.nonzero()
            pome_ig = ig.Graph(
                n=pome_adj.shape[0],
                edges=list(zip(sources, targets)),
                directed=False
            )

            # Run Leiden
            pome_partition = la.find_partition(
                pome_ig,
                la.RBConfigurationVertexPartition
            )
            pome_labels = np.array(pome_partition.membership)
            pome_k = len(pome_partition)
            embedding_df["labels"] = pome_labels
            agg_df = vis_df.join(embedding_df, how="inner")
            
            for method in methods:
                method_df = agg_df[[f"{method}_1", f"{method}_2"]]
                method_labels = agg_df["labels"]
                method_mat = method_df.to_numpy()
                
                pome_db_custom = davies_bouldin_score_custom(method_mat, method_labels, metric="sqeuclidean")
                                                                
                pome_silhouette = silhouette_score(
                    X=method_mat,
                    labels=method_labels,
                    metric="sqeuclidean"
                )
            
                result_dict["run"].append(run)
                result_dict["dim"].append(dim)
                result_dict["method"].append(method)
                result_dict["silhouette"].append(pome_silhouette)
                result_dict["davies_bouldin"].append(pome_db_custom)
                result_dict["dataset"].append(data)

result_df = pd.DataFrame(result_dict)

Processing dataset =  HANCOCK
Processing dimension =  16
Processing dimension =  32
Processing dimension =  64
Processing dataset =  MIMIC
Processing dimension =  16
Processing dimension =  32
Processing dimension =  64
Processing dataset =  TCGA_LUAD
Processing dimension =  16
Processing dimension =  32
Processing dimension =  64


In [22]:
result_df.to_csv("visualization_cluster_preservation_results.csv", index=False)